# 実験009 — Kaggle無料GPUでSee-throughを1枚検証
既存の無料GPUを選択し、Internetを有効にする。無料枠残量とセッション制限をKaggle画面で確認する。GPUは1枚のみ使用。標準方式を先に試し、8GB向け設定は別実行で比較する。モデルライセンスと実行手順は実験報告の environment/README.md 参照。


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

root = Path('/kaggle/working/experiment009')
root.mkdir(exist_ok=True)
print(sys.version, shutil.disk_usage(root))
subprocess.run(['nvidia-smi'], check=True)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Use the upstream-supported Python 3.12 runtime')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
print(torch.__version__, torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU available')
print('Native CUDA BF16:', torch.cuda.is_bf16_supported(including_emulation=False))


In [ ]:
branch = 'experiment/rigging-validation-20261115'
standrig = root / 'StandRig'
official = root / 'see-through'
if not standrig.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, 'https://github.com/keepholdingon1224-source/StandRig', str(standrig)], check=True)
if not official.exists():
    subprocess.run(['git', 'clone', 'https://github.com/shitagaki-lab/see-through.git', str(official)], check=True)
subprocess.run(['git', '-C', str(official), 'checkout', '--detach', 'fc4440db7c076518e313f378e28b1504e9886a95'], check=True)
inputs = standrig / 'source_007/original.png'
assert hashlib.sha256(inputs.read_bytes()).hexdigest() == 'a1388c37f35c19278d346ab4491c8bcc919e1f375b07f312a70ab7e87bbcbcea'
venv = root / 'venv'
subprocess.run([sys.executable, '-m', 'venv', str(venv)], check=True)
py = str(venv / 'bin/python')
subprocess.run([py, '-m', 'pip', 'install', 'torch==2.8.0+cu128', 'torchvision==0.23.0+cu128', 'torchaudio==2.8.0+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([py, '-m', 'pip', 'install', '-r', 'requirements.txt', '-r', 'requirements-inference-bnb.txt', 'psutil==7.2.2'], cwd=official, check=True)
(root / 'pip-freeze.txt').write_bytes(subprocess.check_output([py, '-m', 'pip', 'freeze']))


In [ ]:
import urllib.request
mode, resolution = 'standard', 1280
repos = {
    'standard': ['layerdifforg/seethroughv0.0.2_layerdiff3d', '24yearsold/seethroughv0.0.1_marigold'],
    'nf4': ['24yearsold/seethroughv0.0.2_layerdiff3d_nf4', '24yearsold/seethroughv0.0.1_marigold_nf4'],
}
for repo in repos[mode]:
    with urllib.request.urlopen('https://huggingface.co/api/models/' + repo, timeout=30) as response:
        meta = json.load(response)
    print(repo, meta['sha'], meta.get('cardData', {}).get('license'))
    print(f"https://huggingface.co/{repo}/tree/{meta['sha']}")


In [ ]:
# Paste the immutable revisions from the inspection cell after reviewing the weight terms.
revisions = {}
assert set(revisions) == set(repos[mode]), 'Fill in the two reviewed revisions'
revision_file = root / f'{mode}-revisions.json'
revision_file.write_text(json.dumps(revisions, indent=2))
run_dir = root / f'{mode}-{resolution}'
command = [py, str(standrig / 'experiments/009-run-seethrough.py'), str(official), str(inputs), str(run_dir), str(revision_file), '--mode', mode, '--resolution', str(resolution)]
status = subprocess.run(command).returncode
if run_dir.exists():
    subprocess.run(['sha256sum', '-c', 'SHA256SUMS'], cwd=run_dir, check=True)
    subprocess.run(['tar', '-czf', str(root.parent / f'009-{mode}-{resolution}.tar.gz'), '-C', str(root), run_dir.name, 'pip-freeze.txt', revision_file.name], check=True)
if status:
    raise RuntimeError(f'Inference failed ({status}); recover the logs from Output')
